# RetinaNet — COCO Animal Detection

Full pipeline: setup, data, training, evaluation, visualization.

## 1. Setup

In [ ]:
%cd /content/srv

!pip install -q torch torchvision ultralytics pycocotools PyYAML tqdm scipy matplotlib pillow opencv-python psutil
!pip install -e . -q

## 2. Download COCO 2017

In [ ]:
import os

os.makedirs('data/raw', exist_ok=True)

if not os.path.exists('data/raw/annotations'):
    !wget -q --show-progress -O data/raw/annotations.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q data/raw/annotations.zip -d data/raw/

if not os.path.exists('data/raw/val2017'):
    !wget -q --show-progress -O data/raw/val2017.zip http://images.cocodataset.org/zips/val2017.zip
    !unzip -q data/raw/val2017.zip -d data/raw/

if not os.path.exists('data/raw/train2017'):
    !wget -q --show-progress -O data/raw/train2017.zip http://images.cocodataset.org/zips/train2017.zip
    !unzip -q data/raw/train2017.zip -d data/raw/

print("COCO 2017 downloaded")

## 3. Filter Animal Classes

In [ ]:
from src.dataset.prepare_data import filter_animal_annotations, prepare_yolo_data
from src.utils.utils import load_config, set_seed
from pathlib import Path

config = load_config('configs/default.yaml')
set_seed(42)

animal_classes = config['data']['animal_classes']
print("Target classes:", animal_classes)

filter_animal_annotations(
    raw_dir='data/raw',
    processed_dir='data/processed',
    animal_classes=animal_classes,
    year='2017',
)

yolo_dir = Path('data/processed/yolo')
data_yaml, num_classes, class_names = prepare_yolo_data('data/processed', yolo_dir)
print(f"\nClasses ({num_classes}): {class_names}")
print(f"YOLO data.yaml: {data_yaml}")

## 4. Dataset Statistics

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

stats = {}
for split in ['train', 'val']:
    with open(f'data/processed/{split}/annotations.json') as f:
        data = json.load(f)
    counts = {}
    for ann in data['annotations']:
        cls_id = ann['category_id']
        name = class_names[cls_id - 1]
        counts[name] = counts.get(name, 0) + 1
    stats[split] = counts
    print(f"{split}: {len(data['images'])} images, {sum(counts.values())} annotations")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (split, counts) in zip(axes, stats.items()):
    classes = sorted(counts.keys())
    vals = [counts[c] for c in classes]
    colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))
    ax.barh(classes, vals, color=colors)
    for i, v in enumerate(vals):
        ax.text(v + max(vals)*0.01, i, str(v), va='center', fontsize=9)
    ax.set_title(f'{split.upper()} — distribution by class')
    ax.set_xlabel('Number of annotations')
plt.tight_layout()
plt.savefig('results/plots/dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from PIL import Image
import matplotlib.patches as patches

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

ann_file = 'data/processed/val/annotations.json'
with open(ann_file) as f:
    data = json.load(f)

img_to_anns = {}
for ann in data['annotations']:
    img_to_anns.setdefault(ann['image_id'], []).append(ann)

sample_imgs = [img for img in data['images'] if img['id'] in img_to_anns][:8]

for ax, img_info in zip(axes, sample_imgs):
    img_path = f"data/processed/val/images/{img_info['file_name']}"
    img = Image.open(img_path)
    ax.imshow(img)
    for ann in img_to_anns.get(img_info['id'], []):
        x, y, w, h = ann['bbox']
        cls_name = class_names[ann['category_id'] - 1]
        color = plt.cm.tab10(ann['category_id'] / num_classes)
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 4, cls_name, fontsize=8, color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
    ax.axis('off')
plt.suptitle('Sample Images with Annotations', fontsize=14)
plt.tight_layout()
plt.savefig('results/plots/sample_annotations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.0 Pretrained Model Evaluation (before fine-tuning)

Evaluate the COCO-pretrained RetinaNet on our animal dataset to establish a baseline.

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from src.dataset.dataset import create_dataloaders
from src.evaluation.metrics import evaluate_detections, compute_map_at_range
from src.utils.utils import get_device, cleanup_gpu
from src.dataset.prepare_data import get_dataset_info
from pathlib import Path
from tqdm import tqdm
import torch
import json

device = get_device()
num_classes, class_names = get_dataset_info(config['data']['processed_dir'])

# Load original COCO-pretrained model (91 classes, unmodified head)
pretrained_model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT)
pretrained_model.to(device)
pretrained_model.eval()

# COCO class IDs -> dataset class IDs
coco_to_dataset = {16: 1, 17: 2, 18: 3, 19: 4}  # bird, cat, dog, horse

_, val_loader = create_dataloaders(config, 'retinanet')

predictions = []
ground_truths = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Pretrained eval"):
        images = [img.to(device) for img in images]
        outputs = pretrained_model(images)

        for out in outputs:
            boxes = out['boxes'].cpu()
            labels = out['labels'].cpu()
            scores = out['scores'].cpu()

            mask = torch.tensor([l.item() in coco_to_dataset for l in labels], dtype=torch.bool)
            if mask.any():
                new_labels = torch.tensor([coco_to_dataset[l.item()] for l in labels[mask]])
                predictions.append({'boxes': boxes[mask], 'labels': new_labels, 'scores': scores[mask]})
            else:
                predictions.append({'boxes': torch.zeros((0,4)), 'labels': torch.zeros((0,), dtype=torch.int64), 'scores': torch.zeros((0,))})

        for t in targets:
            ground_truths.append({k: v.cpu() if isinstance(v, torch.Tensor) else v for k, v in t.items()})

pretrained_metrics = evaluate_detections(predictions, ground_truths, num_classes)
range_metrics = compute_map_at_range(predictions, ground_truths, num_classes)
pretrained_metrics['mAP50-95'] = range_metrics['mAP50-95']

per_class = {}
for cls_id, ap in pretrained_metrics.get('per_class_ap', {}).items():
    cls_name = class_names[int(cls_id) - 1]
    per_class[cls_name] = {'ap50': ap}

output_dir = Path('results/retinanet')
output_dir.mkdir(parents=True, exist_ok=True)

save_data = {
    'model': 'retinanet_pretrained',
    'metrics': {
        'mAP50': pretrained_metrics['mAP50'],
        'mAP50-95': pretrained_metrics['mAP50-95'],
        'precision': pretrained_metrics['precision'],
        'recall': pretrained_metrics['recall'],
        'f1': pretrained_metrics['f1'],
        'per_class': per_class,
    }
}

with open(output_dir / 'retinanet_pretrained_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"\n{'='*50}")
print(f"  RetinaNet Pretrained (COCO) Results")
print(f"{'='*50}")
print(f"  mAP@0.5:      {pretrained_metrics['mAP50']:.4f}")
print(f"  mAP@0.5:0.95: {pretrained_metrics['mAP50-95']:.4f}")
print(f"  Precision:     {pretrained_metrics['precision']:.4f}")
print(f"  Recall:        {pretrained_metrics['recall']:.4f}")
print(f"  F1:            {pretrained_metrics['f1']:.4f}")
print(f"\n  Per-class AP@0.5:")
for cls_name, vals in per_class.items():
    print(f"    {cls_name:>12s}: {vals['ap50']:.4f}")

print(f"\nSaved to {output_dir / 'retinanet_pretrained_results.json'}")
del pretrained_model
cleanup_gpu()

## 5. Train RetinaNet

In [ ]:
from src.training.train import train_torchvision_model
from src.evaluation.metrics import evaluate_model
from src.dataset.dataset import create_dataloaders
from src.utils.utils import cleanup_gpu, get_device, get_gpu_memory, print_metrics
from pathlib import Path
import torch

device = get_device()
print("Device:", device)
print("GPU Memory:", get_gpu_memory())

output_dir = Path('results/retinanet')
output_dir.mkdir(parents=True, exist_ok=True)

model, history = train_torchvision_model('retinanet', config, output_dir)

## 6. Evaluate

In [ ]:
_, val_loader = create_dataloaders(config, 'retinanet')

metrics, predictions, ground_truths = evaluate_model(
    model, val_loader, num_classes, device, 'retinanet'
)

print_metrics(metrics, 'RetinaNet')

## 7. Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import csv

# --- Loss Curves ---
log_csv = output_dir / 'training_log.csv'
epochs_list, train_losses, val_losses = [], [], []

with open(log_csv) as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs_list.append(int(row['epoch']))
        train_losses.append(float(row['train_loss']))
        val_losses.append(float(row['val_loss']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_list, train_losses, 'b-o', markersize=4, label='Train Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('RetinaNet — Train Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_list, val_losses, 'r-o', markersize=4, label='Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('RetinaNet — Val Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'train_val_loss.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(epochs_list, train_losses, 'b-o', markersize=4, label='Train Loss')
plt.plot(epochs_list, val_losses, 'r-o', markersize=4, label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('RetinaNet — Training Curves'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Metrics Bar ---
keys = ['mAP50', 'mAP50-95', 'precision', 'recall', 'f1']
values = [metrics.get(k, 0) for k in keys]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

plt.figure(figsize=(10, 6))
bars = plt.bar(keys, values, color=colors)
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=11)
plt.ylim(0, 1.1); plt.title('RetinaNet — Metrics'); plt.ylabel('Score'); plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Per-class AP ---
if metrics.get('per_class_ap'):
    cls_ids = sorted(metrics['per_class_ap'].keys(), key=lambda x: int(x))
    cls_labels = [class_names[int(c)-1] if int(c)-1 < len(class_names) else f'cls_{c}' for c in cls_ids]
    ap_vals = [metrics['per_class_ap'][c] for c in cls_ids]

    plt.figure(figsize=(12, 6))
    bars = plt.barh(cls_labels, ap_vals, color='#2196F3')
    for bar, val in zip(bars, ap_vals):
        plt.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)
    plt.xlim(0, 1.1); plt.xlabel('AP@0.5')
    plt.title('RetinaNet — Per-Class AP'); plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'per_class_ap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- Detection Examples ---
import matplotlib.patches as patches
from PIL import Image
from src.dataset.dataset import create_dataloaders, collate_fn
from torch.utils.data import DataLoader, Subset

_, val_loader = create_dataloaders(config, 'retinanet')
dataset = val_loader.dataset

indices = list(range(min(6, len(dataset))))
subset = Subset(dataset, indices)
loader = DataLoader(subset, batch_size=len(indices), collate_fn=collate_fn)
images, targets = next(iter(loader))

model.eval()
with torch.no_grad():
    imgs_device = [img.to(device) for img in images]
    preds = model(imgs_device)

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(images):
        ax.axis('off'); continue
    img = images[i].cpu().numpy().transpose(1, 2, 0)
    img = np.clip(img * std + mean, 0, 1)
    ax.imshow(img)

    pred = preds[i]
    boxes = pred['boxes'].cpu().numpy()
    labels = pred['labels'].cpu().numpy()
    scores = pred['scores'].cpu().numpy()

    for box, lbl, sc in zip(boxes, labels, scores):
        if sc < 0.3:
            continue
        x1, y1, x2, y2 = box
        idx = int(lbl) - 1
        name = class_names[idx] if 0 <= idx < len(class_names) else f'cls_{lbl}'
        color = plt.cm.tab10(idx / num_classes)
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-4, f'{name} {sc:.2f}', fontsize=8, color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
    ax.axis('off')

plt.suptitle('RetinaNet — Detection Examples', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'detection_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
import json

def _convert(obj):
    import numpy as np_
    if isinstance(obj, (np_.integer,)): return int(obj)
    if isinstance(obj, (np_.floating,)): return float(obj)
    if isinstance(obj, np_.ndarray): return obj.tolist()
    return str(obj)

save_data = {
    'model': 'retinanet',
    'metrics': metrics,
    'history': history,
}

with open(output_dir / 'retinanet_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=_convert)

print("Results saved to", output_dir / 'retinanet_results.json')
cleanup_gpu()